# 🧪 21 · Virtual Staining and Label-Free Imaging
### Predicting a stain you never applied — and knowing when not to believe it

**Part J · Research frontier** · Module 21 of 29 (frontier) · builds on 08, 13, 16, 17

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/21_virtual_staining_label_free_imaging.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (Colab free tier) |
| **Core frameworks** | NumPy, SciPy, PyTorch, Matplotlib |
| **Prerequisites** | Modules 08, 13, 16, 17 |
| **Estimated time** | 70–90 min |
| **Builds on** | Module 08 (deep learning), Module 13 (generative models), Module 16 (denoising) and Module 17 (restoration as an inverse problem) |
| **Research frontier covered** | Virtual / label-free staining — predicting a fluorescent or histological stain from an unstained image, paired vs. unpaired training, and the hallucination risk of trusting a generated stain |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every image pair in this notebook is SIMULATED by `simulate_nuclei_image` and `degrade_image` — there is no real microscopy data, and nothing is downloaded.

## 🎯 Learning objectives

By the end of this module you will be able to:

1. Explain what "virtual" or "label-free" staining means, and why avoiding a real fluorescent or
   histological stain is biologically valuable (phototoxicity, photobleaching, fixation, multiplexing more
   channels than your microscope has detectors for).
2. Distinguish the **paired** training setup (the same field of view imaged both ways) from the harder
   **unpaired** setting, and explain why paired data makes the problem a much more constrained,
   supervised image-to-image regression.
3. Train a small image-to-image network that predicts a target stain-like channel from a degraded,
   label-free-like input, and evaluate it quantitatively (PSNR, a simplified SSIM, and a downstream
   segmentation metric) rather than by eye alone.
4. Demonstrate, with a controlled synthetic experiment, the **hallucination risk**: a network can produce a
   plausible-looking but structurally wrong prediction when asked to extrapolate beyond its training
   distribution — and explain concretely why that is a serious problem for any decision that depends on the
   virtual stain being correct.
5. Describe the kind of validation a real virtual-staining pipeline needs before it is used for anything
   beyond a quick preview.

## 🗺️ Contents

1. The biological question
2. Environment setup
3. What virtual staining predicts, and why it matters
4. Paired vs. unpaired training
5. Training a small image-to-image network on paired synthetic data
6. The hallucination risk: a controlled failure case
7. From toy to real data
8. Key takeaways, checkpoint, and further practice

## 🧫 The biological question

Fluorescent and histological stains are how we normally *see* the structures we care about under a
microscope — but applying them has costs. Fluorescent labeling can be phototoxic and bleach under repeated
imaging, which limits how long you can follow a living cell. Histological staining is destructive and fixes
the tissue, so you only ever see one static timepoint. And every additional fluorescent channel competes for
a limited number of detector channels and excitation wavelengths.

Label-free imaging modalities — plain brightfield, phase contrast, quantitative phase imaging (QPI) — avoid
all of this: they use no stain and barely perturb the sample, so you can image live cells for as long as you
like. The catch is that label-free images are far harder for a human (or a classical algorithm) to interpret
directly: structures that a stain would make obvious are present only as subtle, hard-to-read optical
contrast. Virtual staining asks whether a model can learn to translate that subtle contrast into the stain
image we actually want to look at — without ever applying the real stain to that sample.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 21
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def degrade_image(image: np.ndarray, psf_sigma: float = 1.5, noise_sigma: float = 0.04,
                   seed: int = 0) -> dict:
    """Blur + noise a clean image with a KNOWN point-spread function (the forward model of
    restoration/deconvolution). Returns ``degraded`` and the ``psf`` used."""
    rng = np.random.default_rng(seed)
    psf = gaussian_psf(psf_sigma)
    degraded = blur_image(image, psf)
    degraded = add_gaussian_noise(degraded, noise_sigma, rng).clip(0, None)
    return dict(degraded=degraded, psf=psf, psf_sigma=psf_sigma, noise_sigma=noise_sigma)


def psnr(reference: np.ndarray, estimate: np.ndarray, data_range: float | None = None) -> float:
    """Peak signal-to-noise ratio (dB) between a reference and an estimate; higher is better."""
    data_range = data_range or (reference.max() - reference.min() + 1e-9)
    mse = np.mean((reference - estimate) ** 2)
    if mse <= 1e-12:
        return float("inf")
    return 10 * np.log10(data_range ** 2 / mse)


def simple_ssim(a: np.ndarray, b: np.ndarray, data_range: float | None = None) -> float:
    """A simplified, global (whole-image, not windowed) structural-similarity index: combines
    luminance, contrast and structure agreement; 1.0 = identical. For a full windowed SSIM use
    ``skimage.metrics.structural_similarity``."""
    data_range = data_range or (max(a.max(), b.max()) - min(a.min(), b.min()) + 1e-9)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    mu_a, mu_b = a.mean(), b.mean()
    var_a, var_b = a.var(), b.var()
    cov = ((a - mu_a) * (b - mu_b)).mean()
    return float(((2 * mu_a * mu_b + c1) * (2 * cov + c2)) /
                 ((mu_a ** 2 + mu_b ** 2 + c1) * (var_a + var_b + c2)))


def dice_coefficient(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Dice/F1 overlap of two binary masks: 2|A∩B| / (|A|+|B|)."""
    a, b = mask_a.astype(bool), mask_b.astype(bool)
    inter = (a & b).sum()
    denom = a.sum() + b.sum()
    return float(2 * inter / denom) if denom > 0 else 1.0


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---
## 1 · What virtual staining predicts, and why it matters

Virtual staining ("in silico labeling") trains a network $f_\theta$ to map a
label-free input image $x$ (brightfield, phase contrast, QPI) to a prediction $\hat{y} = f_\theta(x)$ of what
a real stain image $y$ would have shown for that same field of view. Framed this way, it is an **image-to-
image regression** problem: both $x$ and $y$ are images of the same spatial layout, and the network's job is
a pixel-wise (or patch-wise) prediction, not a classification.

*In words:* the label-free image already contains the physical information a stain would reveal — stained
structures scatter and refract light differently than their surroundings even without a fluorophore — but
that information is encoded subtly, as small variations in intensity, phase or texture, rather than as a
bright, specific signal. A trained network learns the (data-specific, learned, not universal) mapping from
"subtle optical signature" to "the stain intensity a biologist would have seen."

Why this matters biologically: one unstained acquisition can, in principle, yield *several* virtual stain
channels — nuclei, cytoplasm, specific organelles — without ever touching the sample, enabling long,
repeated, multi-channel live-cell imaging that a real multi-stain protocol could not sustain.

### 📄 Research connection — from feasibility to failure modes

Christiansen et al. (2018, *Cell*, "In Silico Labeling") and Ounkomol et al. (2018, *Nature Methods*) were
among the first to show, convincingly, that deep networks trained on large paired label-free/fluorescence
datasets could predict several fluorescent channels (nuclei, cell membrane, specific organelles) from a
single transmitted-light image with surprising fidelity. Rivenson et al. (2019, *Nature Biomedical
Engineering*) extended the same idea to **virtual histological staining**, predicting an H&E-like appearance
from label-free tissue autofluorescence images — directly motivated by avoiding the slow, destructive,
reagent-heavy standard histochemistry pipeline. The field has since moved toward **virtual-staining
foundation models**, pretrained across many organisms, stains and microscopes, following exactly the
foundation-model logic of Module 20 — with the same honest caveat: the further a target cell type or stain
is from what the model saw in training, the less you should trust its output without checking.

## 2 · Paired vs. unpaired training

The **paired** setup — the one this notebook uses — requires the *same* field
of view to be imaged twice: once label-free, once with the real stain, carefully registered to each other
(Module 15's registration ideas apply directly if the two acquisitions are not pixel-aligned already). Paired
data turns virtual staining into ordinary supervised regression: minimise a per-pixel loss, typically an
$L_1$ or $L_2$ loss, sometimes combined with a perceptual or adversarial loss (Module 13's GAN ideas),
between $f_\theta(x)$ and the true $y$.

$$\mathcal{L}(\theta) = \mathbb{E}_{(x,y)}\big[\, \| f_\theta(x) - y \|_1 \,\big]$$

The **unpaired** setup has no such correspondence — you have a collection of label-free images and a
*separate* collection of stained images of different cells, with no guarantee any of them show the same
field of view. This is a much harder, less constrained problem: a network can no longer be told "match this
*exact* target," only "produce something that looks statistically like the target domain while preserving
the input's content" (the CycleGAN family of approaches). Unpaired methods are more practically useful when
paired acquisition is genuinely impossible, but they provide a much weaker guarantee that any specific
predicted structure is literally correct — a point Section 4 returns to.

## 3 · Training a small image-to-image network on paired synthetic data

In [ ]:
# ── 3.1 Build a paired synthetic dataset: a label-free-like input, a stain-like target ──
# We SIMULATE a 'ground-truth stain' channel directly with simulate_nuclei_image, and construct a
# 'label-free-like' input by degrading it the way an unstained acquisition loses specific contrast:
# heavier blur (worse effective resolution/contrast transfer) and additive sensor noise, with no
# structure-specific signal boost. This keeps the two images perfectly paired by construction.

def make_pair(seed, shape_classes=("round",), n_cells=10, size=72):
    sim = simulate_nuclei_image(n_cells=n_cells, size=size, shape_classes=shape_classes,
                                 psf_sigma=1.0, noise_sigma=0.02, seed=seed)
    target = sim["image"].astype(np.float32)
    degraded = degrade_image(target, psf_sigma=2.6, noise_sigma=0.05, seed=seed + 1)
    return degraded["degraded"].astype(np.float32), target

N_TRAIN, N_VAL = 48, 8
train_pairs = [make_pair(seed=6000 + i) for i in range(N_TRAIN)]
val_pairs = [make_pair(seed=7000 + i) for i in range(N_VAL)]

x0, y0 = train_pairs[0]
show_image_grid([x0, y0], titles=["label-free-like input (degraded)", "ground-truth stain (target)"], ncols=2)
plt.show()


🔍 **What just happened?** The input and target are pixel-for-pixel aligned by construction (same cell layout, same seed lineage) — exactly the paired-data assumption Section 2 described. In a real experiment this alignment would need careful registration (Module 15); here it is free because both images come from the same simulated ground truth.

In [ ]:
# ── 3.2 A small image-to-image network (encoder–decoder, same spirit as Module 11's U-Net) ──
import torch
import torch.nn as nn

class TinyStainNet(nn.Module):
    def __init__(self, base=12):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, base, 3, padding=1), nn.ReLU(),
                                   nn.Conv2d(base, base, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(nn.Conv2d(base, base * 2, 3, padding=1), nn.ReLU(),
                                         nn.Conv2d(base * 2, base * 2, 3, padding=1), nn.ReLU())
        self.up = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(base * 2, base, 3, padding=1), nn.ReLU(),
                                   nn.Conv2d(base, base, 3, padding=1), nn.ReLU())
        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        b = self.bottleneck(self.pool(e1))
        d1 = self.dec1(torch.cat([self.up(b), e1], dim=1))
        return self.head(d1)

torch.manual_seed(SEED)
stain_net = TinyStainNet()
opt = torch.optim.Adam(stain_net.parameters(), lr=2e-3)

X_train = torch.tensor(np.stack([p[0] for p in train_pairs]))[:, None, :, :].float()
Y_train = torch.tensor(np.stack([p[1] for p in train_pairs]))[:, None, :, :].float()
X_val = torch.tensor(np.stack([p[0] for p in val_pairs]))[:, None, :, :].float()
Y_val = torch.tensor(np.stack([p[1] for p in val_pairs]))[:, None, :, :].float()

train_losses, val_losses = [], []
for epoch in range(60):
    opt.zero_grad()
    pred = stain_net(X_train)
    loss = torch.mean(torch.abs(pred - Y_train))  # L1 loss
    loss.backward()
    opt.step()
    train_losses.append(loss.item())
    with torch.no_grad():
        val_losses.append(torch.mean(torch.abs(stain_net(X_val) - Y_val)).item())

plt.figure(figsize=(4, 3))
plt.plot(train_losses, label="train L1")
plt.plot(val_losses, label="val L1")
plt.legend(); plt.xlabel("epoch"); plt.ylabel("L1 loss"); plt.title("Virtual-staining network")
plt.tight_layout(); plt.show()


In [ ]:
# ── 3.3 Quantitative evaluation: PSNR and a simplified SSIM against the known ground truth ──
def predict(image):
    with torch.no_grad():
        return stain_net(torch.tensor(image[None, None, :, :]).float())[0, 0].numpy()

val_psnrs = [psnr(y, predict(x)) for x, y in val_pairs]
val_ssims = [simple_ssim(y, predict(x)) for x, y in val_pairs]
print(f"validation PSNR : mean {np.mean(val_psnrs):.2f} dB  (higher is better)")
print(f"validation SSIM : mean {np.mean(val_ssims):.3f}  (1.0 = identical)")

x_demo, y_demo = val_pairs[0]
show_image_grid([x_demo, predict(x_demo), y_demo],
                 titles=["label-free-like input", "predicted virtual stain", "ground-truth stain"], ncols=3)
plt.show()


🧪 **Ground-truth check.** Because the target was simulated rather than really
photographed, we have the exact ground-truth stain for every validation image, so PSNR/SSIM here measure
genuine prediction accuracy rather than disagreement between two imperfect real measurements. A real virtual-
staining paper has to work harder for this: it needs a *held-out, truly independent* paired acquisition,
because training and test pairs from the same imaging session can share systematic biases that inflate
apparent accuracy.

✏️ **Your turn.** Increase `psf_sigma` in `make_pair`'s call to `degrade_image` (make the label-free input
even blurrier/noisier) and retrain. How far can you degrade the input before PSNR on the validation set drops
sharply? What does that tell you about how much real optical information the input needs to retain for
virtual staining to work at all?

## 4 · The hallucination risk: a controlled failure case

The network you just trained has only ever seen **round** nuclei. What
happens if you feed it a label-free-like input whose true underlying stain target contains shapes it has
never seen — kidney-shaped, ring-shaped, irregular? A network trained to minimise image-to-image loss has no
built-in mechanism for saying "I don't recognise this input, I refuse to answer" — it will produce *some*
output, and that output can look smooth, confident and plausible while being structurally wrong. This is the
**hallucination risk**: the model fills in what *its training distribution* says should be there, not
necessarily what the (possibly very different) true structure actually is.

In [ ]:
# ── 4.1 Feed the round-shape-only network an out-of-distribution target ──
ood_pairs = [make_pair(seed=8000 + i, shape_classes=("kidney", "ring", "irregular"), n_cells=10)
             for i in range(8)]

ood_psnrs = [psnr(y, predict(x)) for x, y in ood_pairs]
ood_ssims = [simple_ssim(y, predict(x)) for x, y in ood_pairs]
print(f"OOD PSNR : mean {np.mean(ood_psnrs):.2f} dB   (in-distribution was {np.mean(val_psnrs):.2f} dB)")
print(f"OOD SSIM : mean {np.mean(ood_ssims):.3f}   (in-distribution was {np.mean(val_ssims):.3f})")

x_ood, y_ood = ood_pairs[0]
show_image_grid([x_ood, predict(x_ood), y_ood],
                 titles=["label-free-like input (OOD shapes)", "predicted virtual stain\n(hallucinated)",
                         "ground-truth stain"], ncols=3)
plt.show()


🔍 **What just happened?** PSNR/SSIM drop on the OOD set, and — more
importantly — look at *how* the prediction is wrong in the image panel: the network tends to render the
unfamiliar shapes as rounder and smoother than they really are, because "round blob" is the only structure
it ever learned to paint. The prediction does not look obviously broken or noisy — it looks like a
*plausible, clean* stain image, which is exactly what makes hallucination dangerous: a biologist glancing at
the output has no visual cue that it is wrong.

In [ ]:
# ── 4.2 Quantify the hallucination's downstream consequence with a segmentation proxy ──
# If someone segmented the PREDICTED virtual stain instead of a real one, would they get the right answer?
def threshold_segment(image, thr=0.3):
    return (image > thr).astype(np.float32)

id_dice = [dice_coefficient(threshold_segment(predict(x)), threshold_segment(y)) for x, y in val_pairs]
ood_dice = [dice_coefficient(threshold_segment(predict(x)), threshold_segment(y)) for x, y in ood_pairs]
print(f"downstream segmentation Dice, in-distribution virtual stain : {np.mean(id_dice):.3f}")
print(f"downstream segmentation Dice, OOD virtual stain             : {np.mean(ood_dice):.3f}")


✏️ **Your turn.** Repeat the OOD evaluation using a network retrained on *all
six* shape classes (change `make_pair`'s default `shape_classes` for the training set and retrain). Does
broadening the training distribution reduce the hallucination gap, the same remedy Module 20 found for the
specialist-vs-generalist segmentation comparison? What does that suggest about the *real* fix for
hallucination — architecture tricks, or training-data diversity?

**Guarding against over-trusting a virtual stain.** Three practical habits,
all directly motivated by the experiment above:

1. **Always validate on a truly held-out, independently acquired paired set** that resembles the samples you
   actually intend to analyse — not just a random split of the same acquisition session.
2. **Never use a virtual stain as the sole basis for a measurement that drives a decision** (a diagnosis, a
   go/no-go call in a screen) without an occasional real-stain spot-check on a subset of samples.
3. **Report uncertainty, not just a point prediction**, where possible (ensembles or the calibration ideas
   of Module 24 apply directly here) — a confident-looking image is not evidence of a correct one.

## 5 · From toy to real data

To move from this toy image-to-image network to a real virtual-staining
pipeline:

- **Christiansen, E. M. et al. (2018). In Silico Labeling: Predicting Fluorescent Labels in Unlabeled
  Images.** *Cell*, 173(3), 792–803. The foundational demonstration that transmitted-light images alone can
  predict several fluorescent channels.
- **Ounkomol, C. et al. (2018). Label-free prediction of three-dimensional fluorescence images from
  transmitted-light microscopy.** *Nature Methods*, 15, 917–920. Extends the idea to 3-D live-cell imaging.
- **Rivenson, Y. et al. (2019). Virtual histological staining of unlabelled tissue-autofluorescence images
  via deep learning.** *Nature Biomedical Engineering*, 3, 466–477. The histology-side analogue, aimed at
  replacing slow chemical staining protocols.
- Current virtual-staining **foundation-model** efforts extend the Module 20 logic to this task: pretrain
  broadly across organisms, stains and microscopes so that a single released model needs little or no
  fine-tuning on a new lab's data — with the same evaluation discipline (held-out, truly independent test
  data; explicit OOD checks) this module practiced in miniature.

## 📌 Key takeaways

- Virtual staining reframes "see the stain" as image-to-image regression: predict a stain-like target from a
  label-free-like input, trained on paired, registered data.
- Paired training gives a strong, well-defined supervision signal; unpaired training (no matched ground
  truth) is more flexible but gives a far weaker guarantee that any specific predicted structure is correct.
- A trained network has no built-in way to refuse to answer — fed an out-of-distribution input, it produces
  a plausible-looking but structurally wrong prediction: the hallucination risk.
- Quantitative, held-out, genuinely independent evaluation (not a glance at the image) is what separates a
  trustworthy virtual-staining result from an untrustworthy one.

## ✅ Checkpoint

1. Why does paired training data make virtual staining a much better-constrained problem than unpaired
   training?
   <details><summary>Answer</summary>With paired data, the network has an exact, pixel-aligned target for
   every input, so the loss directly measures correctness of each specific prediction. Unpaired data only
   supplies a target *distribution* to match statistically, with no guarantee that any individual predicted
   structure corresponds to the truth for that specific input.</details>

2. In Section 4, why did the OOD prediction look smooth and plausible rather than obviously wrong?
   <details><summary>Answer</summary>The network was only trained to produce the kinds of structures in its
   training targets (round nuclei), so when given an unfamiliar input it renders the closest thing it knows
   how to paint — a clean, confident-looking but incorrect output — rather than an obviously corrupted one.</details>

3. Name two concrete safeguards you would put in place before relying on virtual staining in a real
   experiment.
   <details><summary>Answer</summary>Any two of: validating on a truly independent held-out acquisition;
   periodic real-stain spot-checks; broadening training-data diversity; reporting prediction uncertainty
   rather than a bare point estimate; restricting use to preview/screening rather than final decisions.</details>

## 🧠 Questions & discussion

1. If you could only choose one of "more training images" or "more diverse training images" to reduce
   hallucination risk, which would you choose, and why does Section 4's experiment support that choice?
2. Unpaired virtual staining (e.g. CycleGAN-style approaches) is sometimes the only option when paired
   acquisition is physically impossible. What additional evidence would you want before trusting an unpaired
   model's output, beyond what you would want for a paired model?
3. Virtual staining and the denoising/restoration methods of Modules 16–17 both involve a network "filling
   in" information that was not directly measured. What distinguishes a legitimate restoration from a
   hallucination, if both involve the network adding information beyond what is strictly present in the
   input?

## 🚀 Bonus — going further

1. Train two networks on the same data, one with an L1 loss and one with an L2 (mean-squared-error) loss.
   Compare their PSNR/SSIM and, more importantly, look at their error images (`prediction - target`) — which
   loss produces smoother, blurrier errors?
2. Add an adversarial loss term (a small discriminator network, Module 13-style) to the training objective
   and see whether the predicted images look visually sharper, even if PSNR does not necessarily improve —
   a classic case where a pixel-wise metric and perceptual quality diverge.
3. Simulate a *registration error* between input and target (shift the target by a few pixels before
   training) and observe how much it degrades the trained network's accuracy — motivating why real virtual-
   staining pipelines invest heavily in precise image registration (Module 15) before training.

## 📝 Lab note

```text
Module 21 — Virtual staining and label-free imaging
Date:
Validation PSNR / SSIM (in-distribution):
OOD PSNR / SSIM:
Downstream segmentation Dice, in-distribution vs. OOD:
Where the hallucinated prediction visibly diverged from the ground truth:
Safeguards I would require before trusting this on real data:
```

## 📚 References

1. Christiansen, E. M. et al. (2018). In Silico Labeling: Predicting Fluorescent Labels in Unlabeled Images.
   *Cell*, 173(3), 792–803.
2. Ounkomol, C., Seshamani, S., Maleckar, M. M., Collman, F. & Johnson, G. R. (2018). Label-free prediction
   of three-dimensional fluorescence images from transmitted-light microscopy. *Nature Methods*, 15, 917–920.
3. Rivenson, Y. et al. (2019). Virtual histological staining of unlabelled tissue-autofluorescence images via
   deep learning. *Nature Biomedical Engineering*, 3, 466–477.
4. Zhu, J.-Y., Park, T., Isola, P. & Efros, A. A. (2017). Unpaired Image-to-Image Translation using
   Cycle-Consistent Adversarial Networks (CycleGAN). *ICCV*.
5. Isola, P., Zhu, J.-Y., Zhou, T. & Efros, A. A. (2017). Image-to-Image Translation with Conditional
   Adversarial Networks (pix2pix). *CVPR*.
6. Wang, Z. et al. (2004). Image quality assessment: from error visibility to structural similarity (SSIM).
   *IEEE Transactions on Image Processing*, 13(4), 600–612.

## ➡️ Next

Module 22 moves from predicting one image from another to turning *whole populations* of cell images into
quantitative phenotypic profiles — and confronts a different, equally practical validity problem: batch
effects that can masquerade as real biological signal.